# Ultimate Data Science Challenge

This notebook contains all three parts of the take-home challenge:
1. Exploratory data analysis
2. Experiment and metrics design
3. Predictive modeling

In [ ]:
# Imports and global setup
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)

## Part 1 - Exploratory Data Analysis (logins.json)
Aggregate login counts to 15-minute intervals, visualize the time series, and report key demand patterns and data quality observations.

In [ ]:
# Load login data
data_dir = Path('../data/raw')
logins_path = data_dir / 'logins.json'

with open(logins_path, 'r') as f:
    raw_logins = json.load(f)

logins = pd.DataFrame(raw_logins)
logins.head(), logins.shape

In [ ]:
# Parse timestamps and aggregate to 15-minute bins
logins['login_time'] = pd.to_datetime(logins['login_time'], errors='coerce')

quality_summary = {
    'rows': len(logins),
    'null_login_time': int(logins['login_time'].isna().sum()),
    'duplicate_rows': int(logins.duplicated().sum())
}
quality_summary

In [ ]:
login_counts_15m = (
    logins.dropna(subset=['login_time'])
    .set_index('login_time')
    .resample('15min')
    .size()
    .rename('login_count')
    .to_frame()
)

login_counts_15m.head()

In [ ]:
# Time series visualization
fig, ax = plt.subplots(figsize=(14, 5))
login_counts_15m['login_count'].plot(ax=ax, linewidth=0.8)
ax.set_title('Login Counts in 15-Minute Intervals')
ax.set_xlabel('Time')
ax.set_ylabel('Login Count')
plt.tight_layout()
plt.show()

In [ ]:
# Daily and weekly demand pattern views
tmp = login_counts_15m.copy()
tmp['hour'] = tmp.index.hour
tmp['dayofweek'] = tmp.index.dayofweek  # Monday=0

hourly_profile = tmp.groupby('hour')['login_count'].mean()
dow_profile = tmp.groupby('dayofweek')['login_count'].mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
hourly_profile.plot(kind='bar', ax=axes[0], color='#3b82f6')
axes[0].set_title('Average Logins by Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Avg Login Count')

dow_profile.plot(kind='bar', ax=axes[1], color='#10b981')
axes[1].set_title('Average Logins by Day of Week')
axes[1].set_xlabel('Day of Week (Mon=0)')
axes[1].set_ylabel('Avg Login Count')

plt.tight_layout()
plt.show()

### Part 1 Write-up
- Summarize observed daily/weekly cycles.
- Note peaks/troughs and potential business implications.
- Report data quality findings and any caveats.

## Part 2 - Experiment and Metrics Design
Propose a key success metric and a practical experiment to evaluate toll reimbursement impact on cross-city driver behavior.

### 2.1 Key Metric
Define one primary metric (e.g., share of drivers completing trips in both cities per week) and justify why it best measures behavior change.

### 2.2 Experiment Design
- Population and randomization unit
- Treatment and control definitions
- Experiment duration and power considerations
- Guardrail metrics (earnings, wait time, rider cancellations, etc.)

### 2.3 Statistical Test and Interpretation
- Statistical test(s) and assumptions
- Significance threshold and confidence intervals
- How to interpret possible outcomes
- Caveats and recommendation framework

## Part 3 - Predictive Modeling (ultimate_data_challenge.json)
Build and evaluate a model to predict whether a user is active in month 6.

In [ ]:
# Load retention dataset
users_path = data_dir / 'ultimate_data_challenge.json'
users = pd.read_json(users_path)
users.head(), users.shape

In [ ]:
# Basic cleaning and target construction
users['last_trip_date'] = pd.to_datetime(users['last_trip_date'], errors='coerce')
users['signup_date'] = pd.to_datetime(users['signup_date'], errors='coerce')

snapshot_date = users['last_trip_date'].max()
users['retained'] = (users['last_trip_date'] >= (snapshot_date - pd.Timedelta(days=30))).astype(int)

retention_rate = users['retained'].mean()
retention_rate

In [ ]:
# Feature selection
drop_cols = ['city', 'phone', 'last_trip_date', 'signup_date', 'retained']
feature_candidates = [c for c in users.columns if c not in ['retained']]
X = users[feature_candidates].copy()
y = users['retained']

cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_cols = X.select_dtypes(include=['number', 'datetime64[ns]']).columns.tolist()

# Convert datetime columns to numeric age-like features if present
for c in num_cols:
    if np.issubdtype(X[c].dtype, np.datetime64):
        X[c] = (X[c] - X[c].min()).dt.days

cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))])
preprocess = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])

X_train.shape, X_test.shape

In [ ]:
# Baseline model: Logistic Regression
log_reg = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))
])

log_reg.fit(X_train, y_train)
pred = log_reg.predict(X_test)
proba = log_reg.predict_proba(X_test)[:, 1]

print('ROC-AUC:', round(roc_auc_score(y_test, proba), 4))
print(classification_report(y_test, pred))

In [ ]:
# Alternative model: Random Forest
rf = Pipeline([
    ('prep', preprocess),
    ('model', RandomForestClassifier(n_estimators=400, random_state=42, class_weight='balanced'))
])

rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
rf_pred = rf.predict(X_test)

print('RF ROC-AUC:', round(roc_auc_score(y_test, rf_proba), 4))
print(classification_report(y_test, rf_pred))

### Part 3 Write-up
- Fraction retained in observed cohort.
- Modeling approach and alternatives considered.
- Performance summary and validity concerns.
- Actionable recommendations for operations/marketing.

## Appendix - Full Code
If needed, keep all final cleaned code cells in this notebook and export to script for submission.